# 🤖 Clase 2 · La Revolución de los Transformers — Construyendo **MiniGPT**

**Módulo:** Científico de Datos e Inteligencia Artificial Aplicada · **Unidad 2:** NLP y el Auge de la IA Generativa (LLMs)

---

### 🎯 Objetivos de la clase
Al terminar podrás:
1. Explicar **por qué los Transformers reemplazaron a las RNN** y qué problema resuelve la *atención*.
2. Implementar **self-attention desde cero** (Q, K, V, softmax, máscara causal) con PyTorch.
3. Diferenciar arquitecturas **Encoder-only (BERT)** y **Decoder-only (GPT)** con ejemplos reales.
4. Entender **cómo genera texto un GPT**: predicción del siguiente token, temperatura, top-k y top-p.
5. Convertir un GPT-2 "base" en un **chatbot (MiniGPT)** mediante *fine-tuning supervisado* (SFT).

### 🧭 El proyecto del curso: MiniGPT
Durante la Unidad 2 construimos **nuestro propio "ChatGPT" en miniatura**:

| Clase | Qué le agregamos a MiniGPT |
|---|---|
| **Clase 2 (hoy)** | Entendemos el Transformer por dentro y entrenamos MiniGPT (GPT-2 en español) para que siga un formato de chat |
| Clase 3 | Lo convertimos en un asistente real con un LLM instruct (Qwen3), *prompt engineering*, memoria y streaming |

### ⏱️ Agenda (3 horas)
| Tiempo | Bloque |
|---|---|
| 0:00 – 0:15 | Intro: de embeddings a Transformers |
| 0:15 – 0:35 | **1.** Tokenización subword (BPE) |
| 0:35 – 1:15 | **2.** Self-attention y bloque Transformer desde cero |
| 1:15 – 1:30 | **3.** Mirando dentro de GPT-2 |
| 1:30 – 1:40 | ☕ Pausa |
| 1:40 – 2:00 | **4.** Encoder (BERT) vs Decoder (GPT) |
| 2:00 – 2:25 | **5.** Cómo genera texto un GPT (sampling) |
| 2:25 – 2:55 | **6.** MiniGPT: fine-tuning para chatear |
| 2:55 – 3:00 | Cierre y reto |

---
## 0. Preparación del entorno
Instalamos lo que falte **en el mismo Python que usa este kernel**, importamos librerías y fijamos la configuración.

In [ ]:
# Instala lo que falte en el MISMO Python que usa este kernel
import importlib.util, subprocess, sys

paquetes = {"torch": "torch", "transformers": "transformers", "pandas": "pandas",
            "matplotlib": "matplotlib"}
faltan = [pip for modulo, pip in paquetes.items() if importlib.util.find_spec(modulo) is None]

print("Python del kernel:", sys.executable)
if faltan:
    print("Instalando:", ", ".join(faltan))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *faltan])
    print("✓ Listo. Si algún import falla, reinicia el kernel.")
else:
    print("✓ Todo instalado.")

In [ ]:
import json
import math
import random
import time
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

import transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    StoppingCriteria,
    StoppingCriteriaList,
    get_linear_schedule_with_warmup,
    pipeline,
)

warnings.filterwarnings("ignore")
transformers.logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)

SEED = 42
def fijar_semilla(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

fijar_semilla()

# Dispositivo: GPU NVIDIA (cuda), GPU Apple (mps) o CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"torch {torch.__version__} | transformers {transformers.__version__} | device: {device}")

In [ ]:
# ¿Tienes GPU NVIDIA pero torch no la ve? (pasa si instalaste la versión "+cpu" de PyTorch)
import shutil
if device.type == "cuda":
    print("🚀 Usando GPU:", torch.cuda.get_device_name(0))
elif shutil.which("nvidia-smi"):
    print("⚠️ Tu computador tiene GPU NVIDIA, pero tu PyTorch es la versión solo-CPU:", torch.__version__)
    print("   Instala la versión con CUDA (en una terminal con tu entorno activado) y reinicia el kernel:")
    print("   pip uninstall -y torch")
    print("   pip install torch --index-url https://download.pytorch.org/whl/cu128")
    print("   (si ese comando falla, copia el de https://pytorch.org/get-started/locally/ eligiendo pip + CUDA)")
else:
    print("💻 Sin GPU NVIDIA: entrenaremos en CPU congelando capas (más lento, pero funciona).")

In [ ]:
# ---------------- CONFIGURACIÓN DE LA CLASE ----------------
MODELO_GPT  = "datificate/gpt2-small-spanish"          # GPT-2 small (124 M) preentrenado en Wikipedia en español
MODELO_BERT = "dccuchile/bert-base-spanish-wwm-cased"  # BETO: BERT en español (U. de Chile)

RUTA_DATOS = Path("..") / "datasets" / "dialogos_minigpt.jsonl"
DIR_SALIDA = Path("..") / "modelo" / "miniGPT-es"      # aquí guardamos MiniGPT (lo usamos en la Clase 3)

# Formato de chat que aprenderá MiniGPT
SYSTEM_PROMPT = "Eres MiniGPT, un asistente que responde en español de forma breve y clara."
TAG_USUARIO = "### Usuario:"
TAG_BOT = "### MiniGPT:"

# Hiperparámetros del fine-tuning (pensados para CPU)
CAPAS_CONGELADAS = 6 if device.type == "cpu" else 0   # de 12 bloques; en GPU entrenamos el modelo completo
EPOCHS = 15 if device.type == "cpu" else 25           # con GPU (ej. RTX 4070) 25 épocas tardan ~1-2 min
LEARNING_RATE = 3e-4 if CAPAS_CONGELADAS else 1e-4
BATCH_SIZE = 8

print("Dataset existe:", RUTA_DATOS.exists(), "→", RUTA_DATOS.resolve())

---
## 1. Tokenización subword (BPE): cómo "lee" un GPT

En la Clase 1 tokenizamos por **palabras** (`Tokenizer` de Keras) y vimos el problema de las palabras desconocidas (`<OOV>`).
Los LLMs usan **subpalabras**: el algoritmo **BPE (Byte Pair Encoding)** empieza con caracteres y va **fusionando los pares más frecuentes**.

- Palabras comunes → **1 token** (`casa`, `modelo`)
- Palabras raras → **varios tokens** (`desoxirribonucleico` → `des` + `oxi` + …)
- **Nunca hay OOV**: en el peor caso se parte hasta llegar a bytes.

> GPT-2 trabaja con **bytes**: en la vista "interna" el símbolo **`Ġ`** significa "aquí había un espacio" y las tildes aparecen raras (`Ã³` = `ó`) porque en UTF-8 la `ó` ocupa 2 bytes. Con `tokenizer.decode` lo vemos normal.

In [ ]:
# ✍️ LIVE CODING — cargar el tokenizer y ver tokens + ids
# 1.1 Cargamos el tokenizer de GPT-2 en español y tokenizamos una frase



In [ ]:
# ✍️ LIVE CODING — comparar nº de palabras vs nº de tokens
# 1.2 Palabras vs tokens: las palabras raras se parten en varios pedazos



---
## 2. Self-Attention desde cero

**Problema:** el embedding de "banco" debería cambiar según el contexto. **Solución:** que cada token **mire a los demás** y mezcle su información según qué tan relevantes sean.

Cada token se proyecta en tres vectores (analogía con una búsqueda en una biblioteca):
| Vector | Pregunta que responde | Analogía |
|---|---|---|
| **Query (Q)** | ¿Qué estoy buscando? | Lo que escribes en el buscador |
| **Key (K)** | ¿Qué ofrezco yo? | La etiqueta/título de cada libro |
| **Value (V)** | ¿Qué información aporto? | El contenido del libro |

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

1. $QK^\top$ → **puntaje de afinidad** entre cada par de tokens.
2. $/\sqrt{d_k}$ → escalado para que el softmax no se sature.
3. **softmax** → cada fila se convierte en **pesos que suman 1** ("cuánta atención le pongo a cada token").
4. $\times V$ → cada token sale como una **mezcla ponderada** de la información de los demás.

In [ ]:
# ✍️ LIVE CODING — crear x, W_q, W_k, W_v y calcular Q, K, V
# 2.1 Una frase de juguete: 7 tokens con embeddings aleatorios de dimensión 16



In [ ]:
# ✍️ LIVE CODING — puntajes → softmax → mezcla con V
# 2.2 La fórmula de la atención paso a paso



In [ ]:
# Función auxiliar para dibujar mapas de atención (viene lista)
def plot_atencion(pesos, tokens, titulo="Mapa de atención", ax=None):
    pesos = pesos.detach().cpu().numpy() if torch.is_tensor(pesos) else np.asarray(pesos)
    if ax is None:
        _, ax = plt.subplots(figsize=(0.6 * len(tokens) + 2, 0.6 * len(tokens) + 1.5))
    ax.imshow(pesos, cmap="Purples", vmin=0, vmax=max(pesos.max(), 1e-9))
    ax.set_xticks(range(len(tokens)), tokens, rotation=45, ha="right")
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel("Key (token al que se mira)")
    ax.set_ylabel("Query (token que mira)")
    ax.set_title(titulo)
    if len(tokens) <= 12:
        for i in range(len(tokens)):
            for j in range(len(tokens)):
                v = pesos[i, j]
                ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                        color="white" if v > 0.5 * pesos.max() else "black")
    return ax

plot_atencion(pesos, tokens_toy, "Self-attention SIN máscara (estilo BERT): todos ven a todos")
plt.tight_layout(); plt.show()

### 🎭 La máscara causal: la diferencia entre BERT y GPT
Un modelo que **genera** texto no puede "hacer trampa" mirando las palabras futuras (¡todavía no existen!).
La **máscara causal** pone $-\infty$ en los puntajes de los tokens futuros → después del softmax su peso es **0**.
El mapa queda **triangular**: el token *i* solo ve a los tokens 0…*i*.

In [ ]:
# ✍️ LIVE CODING — crear la máscara triangular y aplicarla
# 2.3 Máscara causal: cada token solo puede mirar hacia atrás



In [ ]:
# 2.4 (Demostración) Efecto del escalado en el softmax con d_k grande
d_grande = 512
q, k = torch.randn(10, d_grande), torch.randn(10, d_grande)
sin_escala = F.softmax(q @ k.T, dim=-1)
con_escala = F.softmax(q @ k.T / math.sqrt(d_grande), dim=-1)
print(f"Peso máximo promedio SIN escalar: {sin_escala.max(dim=-1).values.mean():.3f}  ← casi one-hot (saturado)")
print(f"Peso máximo promedio CON escala : {con_escala.max(dim=-1).values.mean():.3f}  ← reparte la atención")

### 🧠 Atención multi-cabeza y el bloque Transformer
- **Multi-cabeza:** en vez de una sola atención, usamos *h* atenciones pequeñas en paralelo. Cada **cabeza** aprende un tipo de relación distinta (sintaxis, correferencia, posición…). Al final se concatenan.
- **Bloque Transformer (estilo GPT-2):**

```
x ──► LayerNorm ──► Atención multi-cabeza (causal) ──► + (residual)
  ──► LayerNorm ──► Red feed-forward (MLP 4× más ancha, GELU) ──► + (residual) ──► siguiente bloque
```
- **Embeddings de posición:** la atención no sabe el orden de las palabras, así que se **suma** un vector por posición.
- **GPT completo** = embeddings (token + posición) → **N bloques** → LayerNorm → capa lineal que da un puntaje (**logit**) para cada token del vocabulario.

In [ ]:
# ✍️ LIVE CODING — implementar AtencionMultiCabeza (qkv → dividir en cabezas → atención → concatenar)
# 2.5 Atención multi-cabeza como módulo de PyTorch



In [ ]:
# ✍️ LIVE CODING — BloqueTransformer (LayerNorm + atención + residual + MLP) y MiniTransformer
# 2.6 Bloque Transformer y un "MiniTransformer" completo (arquitectura GPT, SIN entrenar)



---
## 3. Mirando dentro de GPT-2 real
Ahora cargamos **GPT-2 small en español** (preentrenado leyendo Wikipedia) y comparamos su tamaño con nuestro MiniTransformer.
Pedimos `output_attentions=True` para **ver los mapas de atención reales** que aprendió.

In [ ]:
# ✍️ LIVE CODING — cargar GPT-2 y comparar su configuración
# 3.1 Cargar GPT-2 en español (attn_implementation="eager" para poder devolver los pesos de atención)



In [ ]:
# ✍️ LIVE CODING — pasar una frase con output_attentions=True y graficar 3 cabezas
# 3.2 Mapas de atención reales de GPT-2



**¿Qué observar?**
- Todos los mapas son **triangulares** → GPT-2 usa máscara causal, igual que la nuestra.
- Muchas cabezas ponen mucho peso en el **primer token** (funciona como "lugar de descanso" cuando no hay nada relevante que mirar).
- Hay cabezas que miran al **token anterior** (diagonal desplazada) y otras que conectan palabras relacionadas (ej. *tenía* → *perro* / *gato*).
- Nadie programó estas reglas: **emergen del entrenamiento**.

---
## ☕ Pausa — 10 minutos
---

## 4. Encoder (BERT) vs Decoder (GPT)

| | **BERT** (Encoder-only) | **GPT** (Decoder-only) |
|---|---|---|
| Atención | **Bidireccional** (ve izquierda y derecha) | **Causal** (solo ve lo anterior) |
| Tarea de entrenamiento | *Masked LM*: adivinar palabras tapadas `[MASK]` | *Causal LM*: predecir el **siguiente** token |
| Sirve para | **Entender**: clasificar, buscar, extraer entidades | **Generar**: chat, resumir, escribir, programar |
| Ejemplos | BERT, BETO, RoBERTa, modelos de embeddings | GPT-2/3/4, Llama, Qwen, Claude |

Probemos ambos con frases en español.

In [ ]:
# ✍️ LIVE CODING — pipeline fill-mask con BETO
# 4.1 BERT rellena huecos usando el contexto de AMBOS lados



In [ ]:
# ✍️ LIVE CODING — GPT continúa el texto con generate (greedy)
# 4.2 GPT solo puede CONTINUAR el texto (no ve lo que viene después)



---
## 5. ¿Cómo genera texto un GPT? Predicción del siguiente token

Un GPT es una función que recibe tokens y devuelve una **distribución de probabilidad sobre el siguiente token**. Generar texto es repetir:

```
prompt → modelo → probabilidades del siguiente token → elegir uno → agregarlo al prompt → repetir
```
Esto se llama generación **autorregresiva**. ChatGPT y Claude escriben así, token por token.

In [ ]:
# ✍️ LIVE CODING — logits de la última posición → softmax → top 10
# 5.1 Las probabilidades del siguiente token



In [ ]:
# ✍️ LIVE CODING — bucle: logits → argmax → concatenar
# 5.2 Generación "greedy" hecha a mano: siempre el token más probable



### 🎲 Estrategias de muestreo
*Greedy* tiende a repetirse y a ser aburrido. Los chatbots **muestrean** con tres perillas:

| Parámetro | Qué hace | Efecto |
|---|---|---|
| **temperatura** | divide los logits: `softmax(logits / T)` | T < 1 → más seguro y repetitivo · T > 1 → más creativo y caótico |
| **top-k** | solo considera los *k* tokens más probables | corta la "cola" de tokens absurdos |
| **top-p** (*nucleus*) | considera los tokens más probables hasta que su probabilidad acumulada llega a *p* | tamaño del corte adaptativo |

In [ ]:
# ✍️ LIVE CODING — muestrear() con temperatura, top-k y top-p + bucle de generación
# 5.3 Implementamos el muestreo nosotros mismos



In [ ]:
# 5.4 Visualizar el efecto de la temperatura sobre la distribución
logits_ult = logits[0, -1]
top_ids = torch.topk(logits_ult, 8).indices
etiquetas = [tokenizer.decode([i]).strip() or repr(tokenizer.decode([i])) for i in top_ids.tolist()]

fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharey=True)
for ax, temp in zip(axes, [0.5, 1.0, 1.8]):
    p = F.softmax(logits_ult / temp, dim=-1)[top_ids].cpu()
    ax.bar(etiquetas, p, color="#7B2CBF")
    ax.set_title(f"temperatura = {temp}")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("probabilidad")
plt.tight_layout(); plt.show()

In [ ]:
# ✍️ LIVE CODING — model.generate con do_sample, temperature, top_k, top_p
# 5.5 Lo mismo con la función oficial: model.generate hace por dentro lo que acabamos de programar



---
## 6. MiniGPT: de modelo base a chatbot (fine-tuning supervisado)

GPT-2 aprendió a **continuar texto de Wikipedia**, no a **conversar**. Si le damos un formato de chat, no sabe qué hacer.
Le enseñaremos un formato fijo:

```
Eres MiniGPT, un asistente que responde en español de forma breve y clara.

### Usuario: ¿Qué es un token?
### MiniGPT: Un token es un pedazo de texto...<|endoftext|>
```
Primero veamos cómo responde **antes** de entrenar.

In [ ]:
# ✍️ LIVE CODING — construir_prompt, criterio de parada y responder()
# 6.1 Formato de chat + función para responder (con criterio de parada)



😅 El modelo base **no sigue el formato**: continúa con cualquier texto parecido a Wikipedia.
Para arreglarlo hacemos **Supervised Fine-Tuning (SFT)**: le mostramos ~90 ejemplos de preguntas y respuestas con el formato correcto. Es exactamente el paso que OpenAI hizo para pasar de GPT-3 a ChatGPT, pero con millones de ejemplos.

In [ ]:
# ✍️ LIVE CODING — leer el JSONL con pandas y separar train/validación
# 6.2 Cargar el dataset de diálogos



### 🎯 El truco del SFT: solo calculamos la pérdida sobre la respuesta
No queremos que el modelo aprenda a escribir el *system prompt* ni la pregunta (eso lo pone el usuario), solo **a responder**.
Por eso, en las etiquetas (`labels`) ponemos **-100** en los tokens del prompt: PyTorch **ignora** esas posiciones al calcular la pérdida.

In [ ]:
# ✍️ LIVE CODING — codificar() con máscara -100 en el prompt
# 6.3 Tokenizar: prompt + respuesta + EOS, con labels = -100 en la parte del prompt



In [ ]:
# ✍️ LIVE CODING — collate con padding dinámico y DataLoaders
# 6.4 Batches con padding dinámico (rellenamos hasta el más largo de cada batch)



In [ ]:
# ✍️ LIVE CODING — requires_grad=False en embeddings y primeros bloques
# 6.5 Congelar las primeras capas (entrena más rápido en CPU y "olvida" menos lo que ya sabía)



In [ ]:
# ✍️ LIVE CODING — AdamW + scheduler + bucle de épocas con train/val loss
# 6.6 Bucle de entrenamiento (el mismo esquema de la Unidad 1: forward → loss → backward → step)



In [ ]:
# 6.7 Curvas de pérdida
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), historial["train"], "o-", label="train", color="#7B2CBF")
plt.plot(range(0, EPOCHS + 1), historial["val"], "s--", label="validación", color="#E0218A")
plt.xlabel("época"); plt.ylabel("pérdida (cross-entropy)"); plt.title("Fine-tuning de MiniGPT")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# ✍️ LIVE CODING — comparar respuestas antes vs después del fine-tuning
# 6.8 ¿Aprendió? Mismas preguntas de antes + preguntas NUEVAS que no estaban en el dataset



In [ ]:
# ✍️ LIVE CODING — save_pretrained del modelo y del tokenizer
# 6.9 Guardar MiniGPT (lo usaremos en la Clase 3)



In [ ]:
# ✍️ LIVE CODING — bucle de chat con input()
# 6.10 ¡A chatear! (escribe 'salir' para terminar)



---
## ✅ Cierre

**Lo que construimos hoy**
1. Vimos que los LLMs leen **tokens subword**, no palabras.
2. Programamos **self-attention** (Q, K, V, softmax, máscara causal), **multi-cabeza** y un **bloque Transformer**.
3. Vimos **BERT** (entiende, bidireccional) vs **GPT** (genera, causal).
4. Programamos la **generación autorregresiva** con temperatura, top-k y top-p.
5. Convertimos GPT-2 en **MiniGPT** con *fine-tuning supervisado*.

**Limitaciones de MiniGPT** (¡y lo que resolveremos en la Clase 3!)
| Problema | Por qué | Solución en la Clase 3 |
|---|---|---|
| Solo sabe lo de sus ~90 ejemplos; inventa en lo demás | Modelo pequeño + pocos datos | Usar un **LLM instruct** (Qwen3) entrenado con millones de instrucciones |
| No recuerda la conversación | Cada pregunta se envía sola | **Memoria**: reenviar el historial con un *chat template* |
| No se le pueden dar instrucciones nuevas | Solo aprendió un formato | **Prompt engineering**: system prompt, few-shot, chain-of-thought |

### 🏋️ Reto / tarea
1. Agrega **20 diálogos nuevos** al JSONL sobre un tema que te guste (fútbol, cocina, tu empresa…) y vuelve a entrenar. ¿Responde bien preguntas parecidas pero no idénticas?
2. Cambia `CAPAS_CONGELADAS` a 0 y a 10. ¿Qué pasa con el tiempo de entrenamiento y con las respuestas?
3. Prueba `responder()` con temperatura 0.2 y 1.3. Describe la diferencia.